In [ ]:
import re
import argparse
import numpy as np
import pandas as pd
from scipy.optimize import linear_sum_assignment

# Подключаем лемматизацию для нормализации слов.
try:
    import pymorphy3
    _morph = pymorphy3.MorphAnalyzer()
    _USE_MORPH = True
except ImportError:
    _morph = None
    _USE_MORPH = False
    print("[WARNING] pymorphy3 not installed; partial matching degraded. "
          "pip install pymorphy3 pymorphy3-dicts-ru")

_lemma_cache: dict = {}
_phrase_cache: dict = {}


def lemmatize_word(w: str) -> str:
    if not _USE_MORPH:
        return w
    if w not in _lemma_cache:
        _lemma_cache[w] = _morph.parse(w)[0].normal_form
    return _lemma_cache[w]


def get_lemma_phrase(phrase: str) -> str:
    if phrase not in _phrase_cache:
        _phrase_cache[phrase] = ' '.join(lemmatize_word(w) for w in phrase.split())
    return _phrase_cache[phrase]


def normalize_text(text) -> str:
    if pd.isna(text):
        return ""
    return re.sub(r'\s+', ' ', str(text).strip().lower())


def normalize_review_id(df: pd.DataFrame) -> pd.DataFrame:
    df = df.dropna(subset=['review_id']).copy()
    df['review_id'] = (
        pd.to_numeric(df['review_id'], errors='coerce').astype('Int64').astype(str)
    )
    return df[df['review_id'] != '<NA>']


# Функции для частичного сопоставления текстов с помощью Rouge-L.
def _lcs_length(a, b):
    n, m = len(a), len(b)
    if n == 0 or m == 0:
        return 0
    dp = [[0] * (m + 1) for _ in range(n + 1)]
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            if a[i - 1] == b[j - 1]:
                dp[i][j] = dp[i - 1][j - 1] + 1
            else:
                dp[i][j] = max(dp[i - 1][j], dp[i][j - 1])
    return dp[n][m]


def _rouge_l_f1(pred_tokens, gold_tokens):
    if not pred_tokens or not gold_tokens:
        return 0.0
    lcs = _lcs_length(pred_tokens, gold_tokens)
    if lcs == 0:
        return 0.0
    p = lcs / len(pred_tokens)
    r = lcs / len(gold_tokens)
    return 2 * p * r / (p + r)


def _dynamic_threshold(gold_len):
    if gold_len <= 2:
        return 0.5
    if gold_len <= 4:
        return 0.6
    return 0.7


def text_match_rouge(pred: str, gold: str) -> bool:
    la, lb = get_lemma_phrase(pred), get_lemma_phrase(gold)
    if la == lb:
        return True
    at, bt = la.split(), lb.split()
    if not at or not bt:
        return False
    return _rouge_l_f1(at, bt) >= _dynamic_threshold(len(bt))   # threshold on GOLD length


def text_match_contain(pred: str, gold: str) -> bool:
    """Old substring matcher. Kept only for a reviewer-facing comparison; NOT the
    method described in the paper."""
    la, lb = get_lemma_phrase(pred), get_lemma_phrase(gold)
    if la == lb:
        return True
    aw, bw = la.split(), lb.split()
    if len(aw) == 1 and len(bw) == 1:
        return False
    if len(aw) == 1:
        return aw[0] in bw
    if len(bw) == 1:
        return bw[0] in aw
    return la in lb or lb in la


TEXT_MATCHERS = {'rouge': text_match_rouge, 'contain': text_match_contain}


# Формируем списки четверок и находим
# оптимальное соответствие между предсказанием и эталоном.
def build_quad_list(rows: pd.DataFrame) -> list:
    return [(
        normalize_text(r['aspect_term']),
        normalize_text(r['opinion']),
        normalize_text(r['aspect']),
        normalize_text(r['sentiment']),
    ) for _, r in rows.iterrows()]


def optimal_match(pred_list, gold_list, match_fn):
    n_pred, n_gold = len(pred_list), len(gold_list)
    if n_pred == 0 and n_gold == 0:
        return 0, 0, 0
    if n_pred == 0:
        return 0, 0, n_gold
    if n_gold == 0:
        return 0, n_pred, 0
    M = np.zeros((n_pred, n_gold), dtype=int)
    for i, p in enumerate(pred_list):
        for j, g in enumerate(gold_list):
            if match_fn(p, g):
                M[i, j] = 1
    ri, ci = linear_sum_assignment(-M)
    tp = int(sum(M[i, j] for i, j in zip(ri, ci)))
    return tp, n_pred - tp, n_gold - tp


SUBTASKS = ['term', 'op', 'asp', 'sent', 'quad']
IDX = {'term': 0, 'op': 1, 'asp': 2, 'sent': 3}


def per_review_counts(gold_df, pred_df, mode, text_match) -> dict:
    gg = normalize_review_id(gold_df).groupby('review_id')
    pg = normalize_review_id(pred_df).groupby('review_id')
    out = {}
    for rid in set(gg.groups) | set(pg.groups):
        gl = build_quad_list(gg.get_group(rid)) if rid in gg.groups else []
        pl = build_quad_list(pg.get_group(rid)) if rid in pg.groups else []
        rec = {}
        if mode == 'exact':
            rec['quad'] = optimal_match(pl, gl, lambda p, g: p == g)
            for k in ['term', 'op', 'asp', 'sent']:
                i = IDX[k]
                rec[k] = optimal_match([q[i] for q in pl], [q[i] for q in gl],
                                       lambda p, g: p == g)
        else:  # partial
            def quad_match(p, g):
                if p[2] != g[2] or p[3] != g[3]:
                    return False
                return text_match(p[0], g[0]) and text_match(p[1], g[1])
            rec['quad'] = optimal_match(pl, gl, quad_match)
            for k in ['term', 'op']:
                i = IDX[k]
                rec[k] = optimal_match([q[i] for q in pl], [q[i] for q in gl], text_match)
            for k in ['asp', 'sent']:
                i = IDX[k]
                rec[k] = optimal_match([q[i] for q in pl], [q[i] for q in gl],
                                       lambda p, g: p == g)
        out[rid] = rec
    return out


def prf_from_counts(tp, fp, fn):
    p = tp / (tp + fp) if (tp + fp) else 0.0
    r = tp / (tp + fn) if (tp + fn) else 0.0
    f = 2 * p * r / (p + r) if (p + r) else 0.0
    return p, r, f


def aggregate(counts, ids, st):
    tp = fp = fn = 0
    for rid in ids:
        if rid in counts:
            t, f, n = counts[rid][st]
            tp += t; fp += f; fn += n
    return tp, fp, fn


# Бутстреп для оценки доверительных интервалов и сравнения моделей.
def bootstrap(counts_by_model, review_ids, B, seed):
    rng = np.random.default_rng(seed)
    ids = np.array(review_ids)
    n = len(ids)
    models = list(counts_by_model)

    point = {m: {st: prf_from_counts(*aggregate(counts_by_model[m], ids, st))
                 for st in SUBTASKS} for m in models}

    boot = {m: {st: np.empty(B) for st in SUBTASKS} for m in models}
    for b in range(B):
        sample = rng.choice(ids, size=n, replace=True)          # shared across models -> paired
        for m in models:
            for st in SUBTASKS:
                boot[m][st][b] = prf_from_counts(*aggregate(counts_by_model[m], sample, st))[2]

    ci = {m: {st: (float(np.percentile(boot[m][st], 2.5)),
                   float(np.percentile(boot[m][st], 97.5))) for st in SUBTASKS} for m in models}

    ref = models[0]
    paired = {}
    for m in models[1:]:
        paired[m] = {}
        for st in SUBTASKS:
            diff = boot[ref][st] - boot[m][st]
            lo, hi = float(np.percentile(diff, 2.5)), float(np.percentile(diff, 97.5))
            p_val = min(2 * min((diff <= 0).mean(), (diff >= 0).mean()), 1.0)
            paired[m][st] = {'delta': point[ref][st][2] - point[m][st][2],
                             'ci': (lo, hi), 'p': float(p_val)}
    return point, ci, paired, ref


# Основная точка входа программы.
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--gold', required=True)
    ap.add_argument('--pred', nargs='+', required=True,
                    help='name=path pairs, e.g. GPT=gpt.csv Yandex=y.csv DeepSeek=d.csv')
    ap.add_argument('--partial', choices=['rouge', 'contain'], default='rouge')
    ap.add_argument('--universe', choices=['union', 'gold'], default='union')
    ap.add_argument('--drop-ids', nargs='*', type=int, default=[])
    ap.add_argument('-B', type=int, default=5000)
    ap.add_argument('--seed', type=int, default=42)
    ap.add_argument('--out', default='bootstrap')
    args = ap.parse_args()

    text_match = TEXT_MATCHERS[args.partial]
    gold = normalize_review_id(pd.read_csv(args.gold))
    models = {}
    for item in args.pred:
        name, path = item.split('=', 1)
        models[name] = normalize_review_id(pd.read_csv(path))

    ids = set(gold['review_id'])
    if args.universe == 'union':
        for df in models.values():
            ids |= set(df['review_id'])
    drop = {str(i) for i in args.drop_ids}
    review_ids = sorted(ids - drop, key=lambda x: int(x))

    print(f"Reviews: {len(review_ids)} (universe={args.universe}, "
          f"dropped={sorted(drop) or 'none'}) | partial={args.partial} | B={args.B}")
    if len(review_ids) != 100:
        print(f"[NOTE] {len(review_ids)} reviews, not 100 — check for stray ids "
              f"(gold max id = {max(int(i) for i in gold['review_id'])}).")

    st_label = {'term': 'Aspect term', 'op': 'Opinion', 'asp': 'Aspect category',
                'sent': 'Sentiment', 'quad': 'Quadruple'}
    mode_label = {'exact': 'STRICT', 'partial': 'PARTIAL'}

    prf_rows, delta_rows = [], []
    for mode in ['exact', 'partial']:
        counts = {m: per_review_counts(gold, df, mode, text_match) for m, df in models.items()}
        point, ci, paired, ref = bootstrap(counts, review_ids, args.B, args.seed)

        print(f"\n{'='*74}\n{mode_label[mode]}\n{'='*74}")
        for st in SUBTASKS:
            print(f"\n{st_label[st]}")
            for m in models:
                p, r, f = point[m][st]
                lo, hi = ci[m][st]
                print(f"  {m:<14} P={p:.4f} R={r:.4f} F1={f:.4f}  95% CI [{lo:.4f}, {hi:.4f}]")
                prf_rows.append({'regime': mode_label[mode], 'subtask': st_label[st],
                                 'model': m, 'P': round(p, 4), 'R': round(r, 4),
                                 'F1': round(f, 4), 'F1_CI_low': round(lo, 4),
                                 'F1_CI_high': round(hi, 4)})
            for m in list(models)[1:]:
                d = paired[m][st]
                lo, hi = d['ci']
                sig = 'yes' if (lo > 0 or hi < 0) else 'no'
                print(f"    Δ({ref}\u2212{m}) = {d['delta']:+.4f}  95% CI [{lo:+.4f}, {hi:+.4f}]"
                      f"  p={d['p']:.4f}  sig={sig}")
                delta_rows.append({'regime': mode_label[mode], 'subtask': st_label[st],
                                   'comparison': f'{ref}-{m}', 'deltaF1': round(d['delta'], 4),
                                   'CI_low': round(lo, 4), 'CI_high': round(hi, 4),
                                   'p': round(d['p'], 4), 'significant': sig})

    pd.DataFrame(prf_rows).to_csv(f'{args.out}_prf_ci.csv', index=False)
    pd.DataFrame(delta_rows).to_csv(f'{args.out}_pairwise.csv', index=False)
    print(f"\nSaved: {args.out}_prf_ci.csv (Tables 3/4 with CI), "
          f"{args.out}_pairwise.csv (pairwise ΔF1)")


if __name__ == '__main__':
    main()
